In [1]:
from xact.log.config import log_manager
from xact.config.gen import config

log_manager.config(level=20)


from xact.llm import llm_client


print(config.XACT_LLM_MODEL)

from xact.llm.llm import LLM


#log.enable(True)
# config.XACT_LLM_MODEL = "qwen2.5:1.5b "

log = log_manager.init(__name__)

llm = LLM("llama3.2:3b")


Feb/09 00:38:17 |   xact.log.config     | INFO     | log initilized
Feb/09 00:38:17 |   xact.log.config     | XACT_STREAM | log streaming initilized
Feb/09 00:38:17 |   xact.config.gen     | INFO     | Config loaded from .env:.env json_path : config.json


qwen2.5:1.5b


In [2]:
log_manager.loggers

{'xact.log.config': <Logger xact.log.config (INFO)>,
 'xact-stream': <Logger xact-stream (INFO)>,
 'xact': <Logger xact (INFO)>,
 'xact.config.gen': <Logger xact.config.gen (INFO)>,
 'xact.llm.llm': <Logger xact.llm.llm (DEBUG)>,
 '__main__': <Logger __main__ (DEBUG)>}

In [3]:
config.XACT_LLM_MODEL = "qwen2.5:1.5b "

In [4]:
log_manager.ext_enable()
log_manager.enable()
llm.model.var = "deepseek-r1:1.5b" 
res = llm.generate("hi ox",generate_format="chat")

print(res)

Feb/09 00:39:07 |     xact.llm.llm      | INFO     | llm out generated


<think>
Okay, so I'm trying to figure out how to help someone who's asking "hi ox." First off, I know that "ox" is an abbreviation for "Oxidation," which is a chemical process where bonds between atoms in a molecule are broken, and new bonds are formed. Oxidation can happen with or without the involvement of oxygen, making it either oxidation (with oxygen) or reduction (without). 

Now, when someone says "hi ox," they're probably asking for help understanding something related to oxidation. But I don't have specific context about what they need assistance with. So, I should consider different angles where someone might ask this question.

One possibility is that the person wants a basic explanation of what oxidation is, its importance in chemistry, or maybe an example of how it works. Another angle could be if they're struggling with balancing chemical equations during oxidation, or perhaps they need help identifying oxidizing agents in a reaction.

I should also think about common mis

In [5]:
import json
import types
from typing import List, Literal, Set, Union

from xact.config.config import ConfigVar
from xact.config.gen import config
from xact.data.data import DataX
from xact.llm.tool import Tool, tool
from xact.vec.vector import VectorModel
from xact.search.search import string_search



route_datalist=List[Union[DataX,Tool,str]]

class RouteData:
    def __init__(self,embed_model=config.XACT_LLM_EMBEDDING_MODEL):
        self.embed_model = embed_model
        self.vecmd = VectorModel(embed_model=embed_model)
        self.data_embd = []
        self.data_list = []
        self.data_embd_str=[]


    def embed(self,data_list:route_datalist,):

        
       
        for data in data_list:
            if isinstance(data,DataX):
                data:DataX = data
                if data.content:
                    self.data_embd_str.append(data.content)
                    self.data_list.append(data)
            elif isinstance(data,Tool):
                data:Tool = data
                fun_schema =  data.get_schema()
                if fun_schema:

                    data_str = json.dumps(fun_schema)
                    name = fun_schema["function"]["name"]
                    description = fun_schema["function"]["description"]
                    embd_str  = f"{name} description : {description} function : [{data_str.strip()}]"
                    
                    self.data_embd_str.append(embd_str)
                    self.data_list.append(data)

            elif isinstance(data,str):
                if data :
                    self.data_embd_str.append(data.strip())
                    self.data_list.append(data)

            elif isinstance(data,types.FunctionType):
                data:Tool = data
                fun_schema =  gen_function_schema(data)
                if fun_schema:

                    data_str = json.dumps(fun_schema)
                    name = fun_schema["function"]["name"]
                    description = fun_schema["function"]["description"]
                    embd_str  = f" {name} description : {description} function : [{data_str.strip()}]"
                    
                    self.data_embd_str.append(embd_str)
                    self.data_list.append(data)

        self.data_embd = self.vecmd.generate(data=self.data_embd_str,model=self.embed_model)


class Router:
    @staticmethod
    def route(prompt:str, route_data:RouteData,weights:set=(45,50,5)):
        
        rout_vec = Router.route_vector(prompt=prompt,route_data=route_data)
        rout_str = Router.route_string(prompt=prompt,route_data=route_data)

        results = []
        for i,data in enumerate(route_data.data_list):
            vidx = rout_vec["idx"].index(i)
            sidx = rout_str["idx"].index(i)

            score = weights[0]*rout_vec["score"][vidx] + weights[1]*0 + weights[2]*rout_str["score"][sidx]
            results.append((i,data,score))
            

        # Sort results by score (descending)
        results.sort(key=lambda x: x[2], reverse=True)

        # Single loop to construct return dictionary
        res = {"idx": [], "data": [], "score": []}
        for i, d, sc in results:
            res["idx"].append(i)
            res["data"].append(d)
            res["score"].append(sc / 100)

        return res
    
    def route_vector(prompt:str, route_data:RouteData):
        vecmd = VectorModel(embed_model=route_data.embed_model)
        return vecmd.search(query_embed=prompt,data=route_data.data_list,data_embed=route_data.data_embd)
    
    
    def route_string(prompt:str, route_data:RouteData):
        return string_search(promt=prompt,data_list=route_data.data_embd_str)
        

    def route_llm(prompt:str, route_data:RouteData):
        pass
    
    
def get_data(year:int,month:int,day:int):
    """return data for given date"""
    return "data"

from xact.llm.tool import gen_function_schema
from xact.utils.gen import gen_datetime, gen_uuid

tools = [
    get_data,
    gen_uuid,
    tool(func=gen_datetime),
    
]
routedata = RouteData()
routedata.embed(data_list=tools+[DataX(content="data")])
routedata

In [6]:
res = Router.route_vector(prompt="give some unique id",route_data=routedata)
res 

{'idx': [1, 3, 2, 0],
 'score': [0.4574908990475241,
  0.452059762787031,
  0.4243961503100083,
  0.41752060200895785],
 'data': [<function xact.utils.gen.gen_uuid()>,
  DataX(uid=UUID('bf83694a-7852-4fb0-ae76-9e182cd13c81'), cid=None, flow_mode='prompt', role='xact', content='data', content_type='str', md_content=None, metadata=None, tags=None, description=None, source=None, embed=None, embed_id=None, created_at=datetime.datetime(2025, 2, 9, 0, 39, 7, 926081), time_triggers=None),
  <function __main__.get_data(year: int, month: int, day: int)>],
 'data_embed': []}

In [7]:
res["data"][2].run()

Feb/09 00:39:24 |    xact.llm.tool      | INFO     | executiong tool : gen_datetime


datetime.datetime(2025, 2, 9, 0, 39, 24, 194066)

In [8]:


@tool()
def my_function(param1: str, param2: int, param3: bool = True) -> str: 
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"


In [9]:
log_manager.loggers["xact.llm.tool"].propagate=False

In [10]:
my_function(1,2)

Feb/09 00:39:24 |    xact.llm.tool      | INFO     | executiong tool : my_function


'result'

In [11]:


from xact.llm.tool import tool


@tool(
    description="mf",
    param_description={
        "param1": "A string parameter.",
        "param2": "An integer parameter.",
        "param3": "An optional boolean parameter. Defaults to True.",
    },
)
def my_f(
    hhh: str, param2: int, param3: bool = True
) -> str:  # Type hints for parameters and return value
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"


def my_ff(
    hhh: str, param2: int, param3: bool = True
) -> str:  # Type hints for parameters and return value
    """
    This function does something.

    Args:
        param1: A string parameter.  (Description of param1)
        param2: An integer parameter. (Description of param2)
        param3: An optional boolean parameter. Defaults to True. (Description of param3)

    Returns:
        A string.
    """
    # ... function code ...
    return "result"

In [12]:
print(my_f(1,2))

Feb/09 00:39:25 |    xact.llm.tool      | INFO     | executiong tool : my_f


result


In [13]:
print(tool(func=my_f)(1,2))

Feb/09 00:39:25 |    xact.llm.tool      | INFO     | executiong tool : my_f


result


In [14]:

from datetime import datetime


tools = [
    gen_function_schema(get_data),
    gen_function_schema(gen_uuid),
    gen_function_schema(gen_datetime),

]

model = "qwen2.5:1.5b"
model = "llama3.2:3b"
# model = "granite3.1-dense:2b"
completion = llm_client.chat.completions.create(
  model=model,
  messages=[{"role": "user", "content": "What the best party [mumbai bangaore chennai]"}],
  tools=tools,
  tool_choice="required",
  temperature=0
)

print(completion.choices[0].message.tool_calls)
for cmp in completion.choices[0]:
    print(cmp )#.choices[0].message.tool_calls)
    print("2222")


[ChatCompletionMessageToolCall(id='call_fqshgr06', function=Function(arguments='{"day":"31","month":"12","year":"2022"}', name='get_data'), type='function', index=0)]
('finish_reason', 'tool_calls')
2222
('index', 0)
2222
('logprobs', None)
2222
('message', ChatCompletionMessage(content='', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_fqshgr06', function=Function(arguments='{"day":"31","month":"12","year":"2022"}', name='get_data'), type='function', index=0)]))
2222


In [15]:
from datetime import date, timedelta

# Get the current date
today = date.today()
print("Today:", today)

# Add one day
tomorrow = today + timedelta(days=1)
print("Tomorrow:", tomorrow)

# Add multiple days
in_three_days = today + timedelta(days=3)
print("In three days:", in_three_days)

# Subtract days (go to the past)
yesterday = today - timedelta(days=1)
print("Yesterday:", yesterday)

# Working with specific dates:
specific_date = date(2024, 1, 1)  # January 1, 2024
next_day = specific_date + timedelta(days=1)
print(f"The day after {specific_date}: {next_day}")

# Example with months/years:
# Note: timedelta only works with days, seconds, microseconds, milliseconds, minutes, hours, and weeks.
# For month or year arithmetic, you need a different approach (see below).

Today: 2025-02-09
Tomorrow: 2025-02-10
In three days: 2025-02-12
Yesterday: 2025-02-08
The day after 2024-01-01: 2024-01-02


In [16]:
from xact.config.gen import Config 



In [17]:
print(res)

{'idx': [1, 3, 2, 0], 'score': [0.4574908990475241, 0.452059762787031, 0.4243961503100083, 0.41752060200895785], 'data': [<function gen_uuid at 0x7fd7ff717060>, DataX(uid=UUID('bf83694a-7852-4fb0-ae76-9e182cd13c81'), cid=None, flow_mode='prompt', role='xact', content='data', content_type='str', md_content=None, metadata=None, tags=None, description=None, source=None, embed=None, embed_id=None, created_at=datetime.datetime(2025, 2, 9, 0, 39, 7, 926081), time_triggers=None), <xact.llm.tool.Tool object at 0x7fd7ff6eb790>, <function get_data at 0x7fd7feb8c5e0>], 'data_embed': []}


In [18]:
config.AUDIO_FORMAT

8